# Notebook 0: pre-evaluation specification selection

Leakage-free selection of the empirical GNAR lag order, observed network, sparsity and neighbourhood depth using the January 2013 to December 2015 validation block.

In [1]:
import importlib, shared_utils
importlib.reload(shared_utils)
from shared_utils import *

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

d = load_data(network="geographic")
Y_full = d["Y_full"].to_numpy()
test_start = d["test_start"]
N = d["N"]
networks = d["networks"]

NETWORKS = ["geographic", "export", "import"]
QUICK = quick_mode()
P_VALUES = [1, 2, 38, 39, 65] if QUICK else list(range(1, 66))
N_VAL = 36
val_start = test_start - N_VAL

# Re-centre using observations strictly before the validation block.
inner_mean = Y_full[:val_start].mean(axis=0, keepdims=True)
Y_inner = Y_full - inner_mean
Y_val = Y_inner[val_start:test_start]
val_dates = pd.to_datetime(d["Y_full"].index[val_start:test_start])

assert Y_val.shape == (36, N)
assert len(val_dates) == 36
assert val_start >= max(P_VALUES)

PRIOR = dict(
    prior_type="minnesota",
    lambda1=MINN["lambda1"],
    lambda2=MINN["lambda2"],
    lambda3=MINN["lambda3"],
    rw_centre=MINN["rw_centre"],
)

def graph_summary(W):
    """Return density and exact stage-2 pair count for a network matrix."""
    A = np.asarray(W) > 0
    np.fill_diagonal(A, False)
    sw2 = compute_stage_weights(W, 2)
    return {
        "density": float(A.sum() / (N * (N - 1))),
        "stage2_pairs": int((sw2[1] > 0).sum()),
    }

def validation_fit(W, p, max_stage):
    """Fit a conjugate GNAR before validation and score the fixed validation block."""
    stages = [max_stage] * p if max_stage else [0] * p
    sw = compute_stage_weights(W, max_stage)
    model = BayesianGNAR(p=p, s=stages, **PRIOR)
    model.fit(Y_inner[:val_start], sw)
    mean, var = model.forecast_test(Y_inner, val_start, N_VAL)
    assert mean.shape == var.shape == Y_val.shape
    assert np.all(np.isfinite(mean)) and np.all(np.isfinite(var)) and np.all(var > 0)
    score = score_forecasts(Y_val, mean, var, val_dates)
    return model, mean, var, score, crps_series(Y_val, mean, var)

print(f"validation: {val_dates[0].date()} to {val_dates[-1].date()}, "
      f"{N_VAL} months x {N} countries")

validation: 2013-01-01 to 2015-12-01, 36 months x 23 countries


## Complete-network lag screen

In [2]:
lag_rows = []
for network in NETWORKS:
    W = networks[network]
    for p in P_VALUES:
        model, _, _, score, _ = validation_fit(W, p, 1)
        lag_rows.append({
            "network": network,
            "p": p,
            "validation_CRPS": score["CRPS"],
            "validation_RMSE": score["RMSE"],
            "effective_training_rows": model.n_obs,
            "mean_coefficients": model.k,
        })

complete_lag_screen = pd.DataFrame(lag_rows)
best_complete = complete_lag_screen.loc[complete_lag_screen["validation_CRPS"].idxmin()]
P_INITIAL = int(best_complete["p"])

ar_rows = []
for p in P_VALUES:
    model, _, _, score, _ = validation_fit(networks["geographic"], p, 0)
    ar_rows.append({
        "p": p,
        "validation_CRPS": score["CRPS"],
        "validation_RMSE": score["RMSE"],
        "effective_training_rows": model.n_obs,
        "mean_coefficients": model.k,
    })
ar_lag_screen = pd.DataFrame(ar_rows)

print("Complete stage-1 minimum by network:")
print(
    complete_lag_screen.loc[
        complete_lag_screen.groupby("network")["validation_CRPS"].idxmin()
    ].sort_values("network").round(6).to_string(index=False)
)
print(
    f"\nGlobal complete-network minimum: {best_complete['network']}, "
    f"p={P_INITIAL}, CRPS={best_complete['validation_CRPS']:.6f}"
)
assert P_INITIAL == 39
assert str(best_complete["network"]) == "geographic"

Complete stage-1 minimum by network:
   network  p  validation_CRPS  validation_RMSE  effective_training_rows  mean_coefficients
    export 39         0.171674         0.309462                     5175                 78
geographic 39         0.170445         0.307003                     5175                 78
    import 39         0.171233         0.308573                     5175                 78

Global complete-network minimum: geographic, p=39, CRPS=0.170445


## Network, density and stage search

In [3]:
K_VALUES = [1, 2, None] if QUICK else list(range(1, N - 1)) + [None]

def candidate_search(p):
    """Evaluate all observed-network, sparsity and admissible stage candidates."""
    rows = []
    for network in NETWORKS:
        for k in K_VALUES:
            Wk = knn_sparsify(networks[network], k)
            g = graph_summary(Wk)
            k_label = "complete" if k is None else str(k)
            for max_stage in [1, 2]:
                if max_stage == 2 and g["stage2_pairs"] == 0:
                    continue
                model, _, _, score, _ = validation_fit(Wk, p, max_stage)
                expected_k = p * (1 + max_stage)
                assert model.k == expected_k
                rows.append({
                    "candidate": f"{network}|k={k_label}|s={max_stage}",
                    "network": network,
                    "k": k_label,
                    "density": g["density"],
                    "max_stage": max_stage,
                    "stage2_exists": bool(g["stage2_pairs"] > 0),
                    "stage2_pairs": g["stage2_pairs"],
                    "mean_coefficients": model.k,
                    "effective_training_rows": model.n_obs,
                    "validation_CRPS": score["CRPS"],
                    "validation_RMSE": score["RMSE"],
                })
    return pd.DataFrame(rows)

search_p39 = candidate_search(P_INITIAL)
best_p39 = search_p39.loc[search_p39["validation_CRPS"].idxmin()]

print(
    f"p={P_INITIAL} winner: {best_p39['network']}, k={best_p39['k']}, "
    f"s={int(best_p39['max_stage'])}, CRPS={best_p39['validation_CRPS']:.6f}"
)
assert best_p39["network"] == "geographic"
assert str(best_p39["k"]) == "2"
assert int(best_p39["max_stage"]) == 2

p=39 winner: geographic, k=2, s=2, CRPS=0.169226


## Geographic k=2 lag refinement

In [4]:
W_geo_k2 = knn_sparsify(networks["geographic"], 2)
geo_graph = graph_summary(W_geo_k2)
assert geo_graph["stage2_pairs"] == 34
print(f"geographic k=2 exact stage-2 pairs: {geo_graph['stage2_pairs']}")

refine_rows = []
refine_losses = {}
for max_stage in [1, 2]:
    for p in P_VALUES:
        model, _, _, score, loss = validation_fit(W_geo_k2, p, max_stage)
        refine_rows.append({
            "p": p,
            "max_stage": max_stage,
            "validation_CRPS": score["CRPS"],
            "validation_RMSE": score["RMSE"],
            "effective_training_rows": model.n_obs,
            "mean_coefficients": model.k,
        })
        refine_losses[f"p{p}_s{max_stage}"] = loss.tolist()

geo_k2_lag_refinement = pd.DataFrame(refine_rows)
best_refined = geo_k2_lag_refinement.loc[
    geo_k2_lag_refinement["validation_CRPS"].idxmin()
]
P_SELECTED = int(best_refined["p"])
S_SELECTED = int(best_refined["max_stage"])

print("Geographic k=2 minimum by stage depth:")
print(
    geo_k2_lag_refinement.loc[
        geo_k2_lag_refinement.groupby("max_stage")["validation_CRPS"].idxmin()
    ].round(6).to_string(index=False)
)
print(
    f"\nJoint geographic k=2 minimum: p={P_SELECTED}, s={S_SELECTED}, "
    f"CRPS={best_refined['validation_CRPS']:.6f}"
)
assert P_SELECTED == 38
assert S_SELECTED == 2

geographic k=2 exact stage-2 pairs: 34


Geographic k=2 minimum by stage depth:
 p  max_stage  validation_CRPS  validation_RMSE  effective_training_rows  mean_coefficients
38          1         0.169400         0.304892                     5198                 76
38          2         0.169161         0.304418                     5198                114

Joint geographic k=2 minimum: p=38, s=2, CRPS=0.169161


## Stage-depth complexity diagnostic

In [5]:
def gaussian_bic(X, y):
    """Gaussian-MLE BIC for a fixed regression design."""
    rank = int(np.linalg.matrix_rank(X))
    if rank != X.shape[1]:
        return np.nan, rank
    beta, *_ = np.linalg.lstsq(X, y, rcond=None)
    resid = y - X @ beta
    n = len(y)
    sigma2 = float(np.mean(resid**2))
    if not np.isfinite(sigma2) or sigma2 <= 0:
        return np.nan, rank
    loglik = -0.5 * n * (np.log(2 * np.pi * sigma2) + 1.0)
    n_parameters = X.shape[1] + 1
    return float(-2 * loglik + n_parameters * np.log(n)), rank

def newey_west_se(x, bandwidth):
    """Newey-West standard error of a sample mean."""
    x = np.asarray(x, dtype=float)
    x = x - x.mean()
    n = len(x)
    lrv = np.sum(x**2) / n
    for lag in range(1, bandwidth + 1):
        cov = np.sum(x[lag:] * x[:-lag]) / n
        lrv += 2 * (1 - lag / (bandwidth + 1)) * cov
    return float(np.sqrt(max(lrv, 0.0) / n))

stage_diag_rows = []
stage_loss = {}
for max_stage in [1, 2]:
    stages = [max_stage] * P_SELECTED
    X, y, names = build_design(
        Y_inner[:val_start], W_geo_k2,
        p=P_SELECTED, stages=stages, mode="global_gnar", h=1
    )
    bic, rank = gaussian_bic(X, y)
    model, _, _, score, loss = validation_fit(W_geo_k2, P_SELECTED, max_stage)
    stage_diag_rows.append({
        "max_stage": max_stage,
        "mean_coefficients": len(names),
        "training_rows": len(y),
        "design_rank": rank,
        "validation_CRPS": score["CRPS"],
        "validation_RMSE": score["RMSE"],
        "BIC": bic,
        "log_marginal_likelihood": model.log_marginal_likelihood(),
    })
    stage_loss[max_stage] = loss

stage_depth_diagnostic = pd.DataFrame(stage_diag_rows)
loss_diff_s1_minus_s2 = stage_loss[1] - stage_loss[2]
HAC_BANDWIDTH = 3
loss_diff_se = newey_west_se(loss_diff_s1_minus_s2, HAC_BANDWIDTH)

print(stage_depth_diagnostic.round(6).to_string(index=False))
print(
    f"\nmonthly CRPS difference s1-s2={loss_diff_s1_minus_s2.mean():+.6f}, "
    f"Newey-West SE={loss_diff_se:.6f}, bandwidth={HAC_BANDWIDTH}"
)
assert stage_depth_diagnostic["training_rows"].nunique() == 1

 max_stage  mean_coefficients  training_rows  design_rank  validation_CRPS  validation_RMSE         BIC  log_marginal_likelihood
         1                 76           5198           76         0.169400         0.304892 3488.582300             -2298.748471
         2                114           5198          114         0.169161         0.304418 3738.798107             -2299.297285

monthly CRPS difference s1-s2=+0.000238, Newey-West SE=0.000142, bandwidth=3


## Selection confirmation

In [6]:
search_p38 = candidate_search(P_SELECTED)
assert set(search_p38["candidate"]) == set(search_p39["candidate"])

best_p38 = search_p38.loc[search_p38["validation_CRPS"].idxmin()]
selection_converged = (
    best_p39["network"] == best_p38["network"]
    and str(best_p39["k"]) == str(best_p38["k"])
    and int(best_p39["max_stage"]) == int(best_p38["max_stage"])
)

assert best_p38["network"] == "geographic"
assert str(best_p38["k"]) == "2"
assert int(best_p38["max_stage"]) == 2
assert selection_converged

selection = {
    "selection_converged": True,
    "p": P_SELECTED,
    "network": "geographic",
    "k": 2,
    "max_stage": 2,
    "stage2_pairs": int(geo_graph["stage2_pairs"]),
    "stages": [2] * P_SELECTED,
    "validation_CRPS": float(best_p38["validation_CRPS"]),
    "validation_RMSE": float(best_p38["validation_RMSE"]),
    "candidate_count": int(len(search_p38)),
    "validation_start": val_dates[0],
    "validation_end": val_dates[-1],
    "validation_months": N_VAL,
    "validation_cells": N_VAL * N,
}

print(
    f"selected specification: p={selection['p']}, network={selection['network']}, "
    f"k={selection['k']}, stage={selection['max_stage']}"
)
print(
    f"validation CRPS={selection['validation_CRPS']:.6f}, "
    f"RMSE={selection['validation_RMSE']:.6f}"
)

selected specification: p=38, network=geographic, k=2, stage=2
validation CRPS=0.169161, RMSE=0.304418


## Save results

In [7]:
save_result("nb0_selection", {
    "selection": selection,
    "complete_lag_screen": complete_lag_screen.to_dict(orient="records"),
    "ar_lag_screen": ar_lag_screen.to_dict(orient="records"),
    "search_p39": search_p39.to_dict(orient="records"),
    "geo_k2_lag_refinement": geo_k2_lag_refinement.to_dict(orient="records"),
    "geo_k2_monthly_crps": refine_losses,
    "stage_depth_diagnostic": stage_depth_diagnostic.to_dict(orient="records"),
    "stage_depth_monthly_loss_difference_s1_minus_s2": loss_diff_s1_minus_s2.tolist(),
    "stage_depth_hac": {
        "bandwidth": HAC_BANDWIDTH,
        "mean_difference": float(loss_diff_s1_minus_s2.mean()),
        "standard_error": loss_diff_se,
    },
    "search_p38": search_p38.to_dict(orient="records"),
    "quick": QUICK,
    "config": run_config(
        p=P_SELECTED, stages=[2] * P_SELECTED,
        network="geographic", k=2, max_stage=2,
        purpose="pre_evaluation_selection",
    ),
})
print("saved nb0_selection")

saved nb0_selection
